 # 02_RLVR_Part2_WalkForward: Expanding Walk-Forward Multi-Seed Committee Training & Bayesian Audit



 - **Engine Architecture:** Expanding Walk-Forward Horizon (Chunks 0..3) with 50/50 Stratified Sampling & Behavioral KL Anchoring.

 - **Ensemble Standard:** Ex-Post Capital Ensembling ($\frac{1}{M} \sum r_p^{(m)}$) across seeds: `[42, 101, 777]`.

 - **Strict Invariant:** Native generation training; zero inter-generation checkpoint migration. Dynamic benchmark configuration (`config.benchmark`).



 ---



 ### 1. `Rew` (Mean Step-Level Reward)

 * **What it is:** The average unscaled reward received by the agent across all parallel vectorized environments per environment step during training rollouts.

 * **Economic Meaning:** In this staggered sleeve engine, the per-step reward is the **daily benchmark-relative excess return** ($r_{p,t} - r_{\text{bm},t}$, net of slippage).

 * **Reading the Values:** Values like `+0.0001` or `+0.0002` indicate that on average during that epoch's rollout buffer, the agent generated approximately **$+1$ to $+2 \text{ bps}$ of daily excess return** across the sampled training episodes.



 ---



 ### 2. `Loss` (Total PPO Objective Loss)

 * **What it is:** The combined loss minimized by the PPO optimizer during mini-batch updates:

   $$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{policy}} + c_1 \mathcal{L}_{\text{value}} - c_2 \mathcal{S}_{\text{entropy}} + \beta \mathcal{D}_{\text{KL}}(\pi_{\text{new}} \parallel \pi_{\text{anchor}})$$

 * **Reading the Values:** Negative loss values (e.g., `-0.0524`) occur because the PPO surrogate policy objective $\mathbb{E}\left[\min(r_t(\theta)\hat{A}_t, \text{clip}(r_t(\theta))\hat{A}_t)\right]$ is framed as maximizing advantage-weighted log-probabilities (hence minimizing negative objective). A steadily stabilizing loss indicates non-divergent policy updates.



 ---



 ### 3. `EV` (Explained Variance of the Critic)

 * **What it is:** The variance of the discounted future returns explained by the Critic (Value Network):

   $$\text{EV} = 1 - \frac{\text{Var}(y_{\text{target}} - V_{\phi}(s))}{\text{Var}(y_{\text{target}})}$$

 * **Why it matters:** This is the primary health diagnostic of reinforcement learning:

   * **$\text{EV} \to 1.0$:** Perfect value predictions (critic understands portfolio trajectory).

   * **$\text{EV} \in [0.40, 0.75]$:** **Healthy, realistic financial range** (the agent reliably predicts expected sleeve return without overfitting noise).

   * **$\text{EV} \le 0.0$:** Critic is worse than guessing the historical mean.



 ---



 ### 4. `Val Sh` (Validation Annualized Sharpe Ratio)

 * **What it is:** The annualized Sharpe ratio achieved by the policy on the **unseen validation slice** (`cal_val`, the trailing 252 trading days prior to deployment):

   $$\text{Sharpe}_{\text{val}} = \frac{\mu(r_{p,t})}{\sigma(r_{p,t})} \times \sqrt{252}$$



 ---



 ### 5. `Excess` (Cumulative Benchmark-Relative Return)

 * **What it is:** The total compounded excess return over `config.benchmark` across the entire 252-day validation period:

   $$\text{Excess} = \prod_{t=1}^{T_{\text{val}}} (1 + r_{p,t}) - \prod_{t=1}^{T_{\text{val}}} (1 + r_{\text{bm},t})$$



 ---



 ### 6. `⭐ [NEW BEST! Fit: ...]` (Composite Fitness Objective)

 * **What it is:** The authoritative criterion used to determine whether the current epoch's weights should be saved as the champion model (`model_chunk3_s{seed}_champion.pt`):

   $$\text{Fitness} = \text{QuantUtils.compute\_composite\_fitness}(\text{Excess Return}, \text{Information Ratio})$$

 * **Why not optimize on Sharpe alone?** A naive high-beta policy can produce a high Sharpe simply by holding market risk during a bull market. The **Composite Fitness** balances total excess spread and risk-adjusted consistency ($\text{IR}$).

 * **`P:X/15` (Patience Counter):** If 15 consecutive epochs elapse without beating the current best fitness, training terminates early to prevent overfitting to the validation slice.



 ---



 ### 💡 Quant Nugget of Wisdom: Diagnosing Critic Decoupling

 If you ever observe `Val Sh` collapsing while `EV` remains high ($>0.80$), the value network is overfitting to past trends while the actor is taking excessive risk. Conversely, when `EV` rises in tandem with `Val Sh` and `Fit`, the actor's actions and the critic's value estimates are in **healthy, mutually reinforcing alignment**.

In [1]:
# CELL 1: Setup, Environment Configuration & Pre-Flight Tripwire
import multiprocessing
import os
from pathlib import Path
import shutil
import sys
import pytest
import torch

# 1. Thread Locks & Cache Resolution Settings
os.environ["CACHE_LOOKBACK"] = "41"
# Cache start date: set to '1998-01-01' for baseline cache (alpha_cache_41d_1998.parquet).
# Target: switch to '2015-01-01' once alpha_cache_41d_2015.parquet is built (Chunk 0 begins 2016-01-04).
os.environ["CACHE_START_DATE"] = "2015-01-01"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
torch.set_num_threads(1)


# 2. Mount Google Drive or Set Local Directory
RUNNING_IN_COLAB = "google.colab" in sys.modules
if RUNNING_IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    rl_root = Path(
        "/content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v3"
    )
else:
    rl_root = Path(r"C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3")

if str(rl_root) not in sys.path:
    sys.path.append(str(rl_root))
os.chdir(rl_root)

# 3. Tripwire Verification Gate
print("🛡️ Running pre-flight unit tests...")
ret_code = pytest.main(
    ["-x", "tests/test_walk_forward.py", "tests/test_mtm_invariants.py", "-q"]
)
if ret_code != 0:
    raise SystemError(
        "❌ EXECUTION ABORTED: Invariant tests failed! Fix code before training."
    )
print("✅ ALL TESTS PASSED. Device:", "cuda" if torch.cuda.is_available() else "cpu")

Mounted at /content/drive
🛡️ Running pre-flight unit tests...
...............                                                          [100%]
15 passed in 37.32s
✅ ALL TESTS PASSED. Device: cpu


 ## CELL 2: Launch Walk-Forward Multi-Seed Committee Training



 Dynamically resolves the latest existing generation index across output artifacts and increments by 1.

 Runs an independent multi-seed committee walk-forward pipeline across all chunks with native checkpoint isolation.

In [2]:
# CELL 2: Launch Walk-Forward Multi-Seed Committee Training
import re
import time
from pathlib import Path
from run_walk_forward import run_walk_forward_pipeline


def detect_next_generation(output_dir: Path | str = "output") -> int:
    """
    Scans the output directory hierarchy for existing generation artifacts
    (e.g., walk_forward_gen{G}, blotter_*_gen{G}.parquet, run_metadata_gen{G}.json)
    and returns max(generation) + 1. Defaults to 1 if no artifacts are discovered.
    """
    out_path = Path(output_dir)
    if not out_path.exists():
        return 1

    gen_pattern = re.compile(
        r"(?:^|[\W_])(?:gen|generation)_?(\d+)(?:[\W_]|$)", re.IGNORECASE
    )
    discovered_gens = set()

    for p in out_path.rglob("*"):
        match = gen_pattern.search(p.name)
        if match:
            try:
                discovered_gens.add(int(match.group(1)))
            except ValueError:
                continue

    if not discovered_gens:
        return 1

    latest_gen = max(discovered_gens)
    return latest_gen + 1


# 1. Pipeline Execution Configuration
# Set OVERRIDE_GEN to an integer to force a specific generation; leave None for auto-increment.
OVERRIDE_GEN = None
COMMITTEE_SEEDS = [42, 101, 777]

output_path = rl_root / "output" if "rl_root" in globals() else Path("output")

if OVERRIDE_GEN is not None:
    target_generation = int(OVERRIDE_GEN)
    print(
        f"⚠️ MANUAL OVERRIDE: Running explicitly assigned Generation {target_generation}"
    )
else:
    target_generation = detect_next_generation(output_path)
    print(
        f"🔎 Auto-detected ground truth state: Launching Generation {target_generation}"
    )

# 2. Execution Run
start_time = time.time()
print(
    f"🚀 Launching Generation {target_generation} Walk-Forward Committee across seeds: {COMMITTEE_SEEDS}"
)

pipeline_results = run_walk_forward_pipeline(
    generation=target_generation,
    seeds=COMMITTEE_SEEDS,
)

elapsed = time.strftime("%H:%M:%S", time.gmtime(time.time() - start_time))
print(f"\n🎉 Generation {target_generation} Complete in {elapsed}!")
print(f"📊 Pipeline Verdict: {pipeline_results.get('verdict', 'N/A')}")
print(f"💾 Authoritative Continuous Blotter: {pipeline_results.get('blotter_path')}")

🔎 Auto-detected ground truth state: Launching Generation 30
🚀 Launching Generation 30 Walk-Forward Committee across seeds: [42, 101, 777]
🚀 Initializing Gen 30 Walk-Forward Ex-Post Ensemble Engine (Seeds: [42, 101, 777]) on cpu

🔄 EXECUTING CHUNK 0: Chunk_0_Base_Pretraining | Train: 1563d | Deploy: 2022-03-25 -> 2023-03-24 (251d)
   ⚙️ Seed 42: Processing Chunk 0...
      [Chunk 0|s42] Ep 01/90 | Rew: -0.0001 | Loss: -0.0434 | EV: -4.494 | Val Sh: 0.477 | Excess: -6.88% | 12.8s | 🔥[WARMUP]
      [Chunk 0|s42] Ep 02/90 | Rew: -0.0001 | Loss: -0.0832 | EV: -1.104 | Val Sh: 0.425 | Excess: -7.69% | 12.8s | 🔥[WARMUP]
      [Chunk 0|s42] Ep 03/90 | Rew: -0.0001 | Loss: -0.0867 | EV: -0.412 | Val Sh: 0.584 | Excess: -5.22% | 13.2s | 🔥[WARMUP]
      [Chunk 0|s42] Ep 04/90 | Rew: -0.0000 | Loss: -0.0877 | EV: 0.192 | Val Sh: 0.544 | Excess: -5.94% | 13.2s | 🔥[WARMUP]
      [Chunk 0|s42] Ep 05/90 | Rew: -0.0001 | Loss: -0.0875 | EV: 0.209 | Val Sh: 0.552 | Excess: -5.78% | 13.5s | 🔥[WARMUP]
   

In [3]:
# CELL 3: Disconnect Cloud Runtime (Optional)
from google.colab import runtime

try:
    runtime.unassign()
    print("Runtime unassigned. Cloud session disconnected.")
except Exception as e:
    print(f"Local or non-Colab execution detected ({e}). Runtime remains active.")

Runtime unassigned. Cloud session disconnected.
